# Homework 4 – Functions II


In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from random import randint
from math import sqrt

## Q1. Import the data as a pd.DataFrame object. Use one or more functions from the pandas library to split the data into three unique datasets. The first should include columns that are factors only (i.e., categorical data), the second should include columns that are numeric only, and the third should include columns with logical/boolean variables only.

In [2]:
housing = pd.read_csv("Housing_prices_data.csv")
housing.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
factors_df = housing.select_dtypes(include=["object"]).copy()
numeric_df = housing.select_dtypes(include=[np.number]).copy()
logical_df = housing.select_dtypes(include=["bool"]).copy()

print("Full dataset shape:", housing.shape)
print("Factors-only shape:", factors_df.shape)
print("Numeric-only shape:", numeric_df.shape)
print("Logical-only shape:", logical_df.shape)

factors_df.head()

Full dataset shape: (1460, 81)
Factors-only shape: (1460, 42)
Numeric-only shape: (1460, 38)
Logical-only shape: (1460, 1)


,MSZoning,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,...,GarageType,GarageFinish,GarageQual,GarageCond,PavedDrive,PoolQC,Fence,MiscFeature,SaleType,SaleCondition
0,RL,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal
1,RL,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal
2,RL,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal
3,RL,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,...,Detchd,Unf,TA,TA,Y,NaN,NaN,NaN,WD,Abnorml
4,RL,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,...,Attchd,RFn,TA,TA,Y,NaN,NaN,NaN,WD,Normal


## Q2. Using the second dataset from question #1, for each numeric variable use the apply function from the pandas library to return the mean, median and standard deviation (SD) of all numeric variables.

In [5]:
numeric_summary = numeric_df.apply(["mean", "median", "std"]).T
numeric_summary.head()

,mean,median,std
Id,730.500000,730.5,421.610009
MSSubClass,56.897260,50.0,42.300571
LotFrontage,70.049958,69.0,24.284752
LotArea,10516.828082,9478.5,9981.264932
OverallQual,6.099315,6.0,1.382997


In [6]:
numeric_summary.tail()

,mean,median,std
PoolArea,2.758904,0.0,40.177307
MiscVal,43.489041,0.0,496.123024
MoSold,6.321918,6.0,2.703626
YrSold,2007.815753,2008.0,1.328095
SalePrice,180921.195890,163000.0,79442.502883


## Q3. Using the second dataset from question #1, create a new dataset that only includes the variable indicating sales price ("SalePrice"). Search for a categorical variable which has between 2-4 categories in the full dataset that you think may be related to the sales price of houses. Give the mean, median and SD for SalePrice based on the different groups of the categorical variables. Does the average price seem to vary by the different levels of the categorical factor you chose?

I chose **PavedDrive**, which has three categories (`Y`, `P`, `N`) and is plausibly related to house value.

In [7]:
saleprice_only = numeric_df[["SalePrice"]].copy()
saleprice_only.head()

,SalePrice
0,208500
1,181500
2,223500
3,140000
4,250000


In [8]:
saleprice_by_paveddrive = (
    housing.groupby("PavedDrive")["SalePrice"]
    .agg(["mean", "median", "std"])
    .reset_index()
)

saleprice_by_paveddrive.head()

,PavedDrive,mean,median,std
0,N,115039.122222,111000.0,44352.523309
1,P,132330.000000,132250.0,33503.030228
2,Y,186433.973881,168500.0,79665.503047


In [9]:
saleprice_by_paveddrive.tail()

,PavedDrive,mean,median,std
0,N,115039.122222,111000.0,44352.523309
1,P,132330.000000,132250.0,33503.030228
2,Y,186433.973881,168500.0,79665.503047


**Answer:** Yes, the average sale price appears to vary across the levels of PavedDrive. Homes with paved driveways (`Y`) have the highest average sale price, homes with partial paving (`P`) are in the middle, and homes with no paved driveway (`N`) have the lowest average sale price.

## Q4. Regress "SalePrice" on "LotArea", "OverallQual", "OverallCond" with the statsmodels library. Use the syntax below to replicate the output shown you are able to estimate this regression model.

In [10]:
model_full = smf.ols("SalePrice ~ LotArea + OverallQual + OverallCond", data=housing).fit()
model_full.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SalePrice   R-squared:                       0.659
Model:                            OLS   Adj. R-squared:                  0.658
Method:                 Least Squares   F-statistic:                     935.9
Date:                Thu, 02 Apr 2026   Prob (F-statistic):               0.00
Time:                        02:50:51   Log-Likelihood:                -17760.
No. Observations:                1460   AIC:                         3.553e+04
Df Residuals:                    1456   BIC:                         3.555e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept   -1.022e+05   8633.073    -11.832      0.000   -1.19e+05   -8.52e+04
LotArea         1.4503      0.123     11.831      0.000       1.210       1.691
OverallQual   4.43e+04    888.434     49.860      0.000    4.26e+04     4.6e+04
OverallCond  -423.6737   1097.973     -0.386      0.700   -2577.451    1730.104
==============================================================================
Omnibus:                      574.483   Durbin-Watson:                   1.966
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             7330.883
Skew:                           1.468   Prob(JB):                         0.00
Kurtosis:                      13.578   Cond. No.                     1.04e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.04e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## Q5. Write a for loop that loops through each level of the categorical factor you chose in Q3, where each iteration of the for loop subsets your data for one group of the categorical variable at a time, and then fits the same regression model specified above in Q4 *k* times, where *k* refers to the number of unique categories. On each iteration of your for loop, store the model.summary() information as a new element in a list object. Once your for loop is done running, print the list. There should be *k* regressions which you estimated. Does the effect of LotArea on SalePrice vary within the different levels of the categorical factor you chose? What is another, more common way to answer a question like this (i.e., a question about moderation or effect heterogeneity)?

In [11]:
model_summaries = []
coef_rows = []

for level in sorted(housing["PavedDrive"].dropna().unique()):
    subset = housing[housing["PavedDrive"] == level]
    model = smf.ols("SalePrice ~ LotArea + OverallQual + OverallCond", data=subset).fit()
    model_summaries.append(model.summary())
    coef_rows.append({
        "PavedDrive": level,
        "LotArea_coef": model.params["LotArea"],
        "LotArea_se": model.bse["LotArea"],
        "LotArea_pvalue": model.pvalues["LotArea"],
        "n": int(model.nobs)
    })

print(model_summaries)

[<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
Dep. Variable:              SalePrice   R-squared:                       0.500
Model:                            OLS   Adj. R-squared:                  0.483
Method:                 Least Squares   F-statistic:                     28.70
Date:                Thu, 02 Apr 2026   Prob (F-statistic):           5.92e-13
Time:                        02:50:58   Log-Likelihood:                -1059.0
No. Observations:                  90   AIC:                             2126.
Df Residuals:                      86   BIC:                             2136.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
I

In [12]:
coef_comparison = pd.DataFrame(coef_rows)
coef_comparison.head()

,PavedDrive,LotArea_coef,LotArea_se,LotArea_pvalue,n
0,N,2.394493,0.482076,3.415476e-06,90
1,P,3.050797,0.651206,7.723031e-05,30
2,Y,1.410988,0.125756,5.545305e-28,1340


In [13]:
coef_comparison.tail()

,PavedDrive,LotArea_coef,LotArea_se,LotArea_pvalue,n
0,N,2.394493,0.482076,3.415476e-06,90
1,P,3.050797,0.651206,7.723031e-05,30
2,Y,1.410988,0.125756,5.545305e-28,1340


**Answer:** The estimated effect of LotArea on SalePrice does vary across the PavedDrive groups, but the size of the difference should be interpreted together with the standard errors, not by coefficient values alone. A more common way to test moderation or effect heterogeneity is to fit a single regression with interaction terms, such as `SalePrice ~ LotArea * C(PavedDrive) + OverallQual + OverallCond`.

## Q6.  Using the first dataset from question #1, for each categorical variable use a for loop to print a frequency and relative frequency table for every variable. To make your output readable, for each variable you should print the following in this order, placing the variable name where you see blanks below.

In [14]:
for col in factors_df.columns:
    print(f"Frequency table for {col} ============")
    print(factors_df[col].value_counts(dropna=False))
    print()
    print(f"Relative frequency table for {col} ============")
    print(factors_df[col].value_counts(dropna=False, normalize=True))
    print("\n\n")

Frequency table for MSZoning ============
MSZoning
RL         1151
RM          218
FV           65
RH           16
C (all)      10
Name: count, dtype: int64

Relative frequency table for MSZoning ============
MSZoning
RL         0.788356
RM         0.149315
FV         0.044521
RH         0.010959
C (all)    0.006849
Name: proportion, dtype: float64



Frequency table for Street ============
Street
Pave    1454
Grvl       6
Name: count, dtype: int64

Relative frequency table for Street ============
Street
Pave    0.99589
Grvl    0.00411
Name: proportion, dtype: float64



Frequency table for Alley ============
Alley
NaN     1369
Grvl      50
Pave      41
Name: count, dtype: int64

Relative frequency table for Alley ============
Alley
NaN     0.937671
Grvl    0.034247
Pave    0.028082
Name: proportion, dtype: float64



Frequency table for LotShape ============
LotShape
Reg    925
IR1    484
IR2     41
IR3     10
Name: count, dtype: int64

Relative frequency table for LotShape ==========

## Q7. Do the same thing as number 6, but with a while loop this time.

In [15]:
i = 0
factor_cols = list(factors_df.columns)

while i < len(factor_cols):
    col = factor_cols[i]
    print(f"Frequency table for {col} ============")
    print(factors_df[col].value_counts(dropna=False))
    print()
    print(f"Relative frequency table for {col} ============")
    print(factors_df[col].value_counts(dropna=False, normalize=True))
    print("\n\n")
    i += 1

Frequency table for MSZoning ============
MSZoning
RL         1151
RM          218
FV           65
RH           16
C (all)      10
Name: count, dtype: int64

Relative frequency table for MSZoning ============
MSZoning
RL         0.788356
RM         0.149315
FV         0.044521
RH         0.010959
C (all)    0.006849
Name: proportion, dtype: float64



Frequency table for Street ============
Street
Pave    1454
Grvl       6
Name: count, dtype: int64

Relative frequency table for Street ============
Street
Pave    0.99589
Grvl    0.00411
Name: proportion, dtype: float64



Frequency table for Alley ============
Alley
NaN     1369
Grvl      50
Pave      41
Name: count, dtype: int64

Relative frequency table for Alley ============
Alley
NaN     0.937671
Grvl    0.034247
Pave    0.028082
Name: proportion, dtype: float64



Frequency table for LotShape ============
LotShape
Reg    925
IR1    484
IR2     41
IR3     10
Name: count, dtype: int64

Relative frequency table for LotShape ==========

## Q8. Using the Rocket class which we created as part of the week 7 class activity, we are going to create a new class called shuttle. The shuttle class should inherit all of Rocket's characteristics but has one additional parameter: flights_completed which measures the number of flights the shuttle has completed.

In [16]:
class Rocket:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distance_from(self, other):
        return round(sqrt((self.x - other.x)**2 + (self.y - other.y)**2), 2)


class Shuttle(Rocket):
    def __init__(self, x, y, flights_completed):
        super().__init__(x, y)
        self.flights_completed = flights_completed

In [17]:
my_shuttle = Shuttle(12, 8, 5)
print(f"The shuttle is located at ({my_shuttle.x}, {my_shuttle.y}), and has recorded {my_shuttle.flights_completed} flights")

The shuttle is located at (12, 8), and has recorded 5 flights


## Q9. Write a single for loop which randomly creates 11 rockets and 11 shuttles using randomly generated integers. Store this output in two separate lists. Once you have your data, use for loops to print the messages you see below. Note you can use the randint() function from the random module (which you can import with the command from random import randint) to randomly generate the characteristics of your Rockets and Shuttles.

In [18]:
rockets = []
shuttles = []

for _ in range(11):
    rockets.append(Rocket(randint(0, 100), randint(0, 100)))
    shuttles.append(Shuttle(randint(0, 100), randint(0, 100), randint(0, 10)))

print("Number of rockets:", len(rockets))
print("Number of shuttles:", len(shuttles))

Number of rockets: 11
Number of shuttles: 11


In [19]:
for i, shuttle in enumerate(shuttles):
    print(f"Shuttle {i} has completed {shuttle.flights_completed} flights.")

Shuttle 0 has completed 10 flights.
Shuttle 1 has completed 8 flights.
Shuttle 2 has completed 5 flights.
Shuttle 3 has completed 7 flights.
Shuttle 4 has completed 6 flights.
Shuttle 5 has completed 6 flights.
Shuttle 6 has completed 3 flights.
Shuttle 7 has completed 0 flights.
Shuttle 8 has completed 1 flights.
Shuttle 9 has completed 4 flights.
Shuttle 10 has completed 1 flights.


In [20]:
first_shuttle = shuttles[0]

for i, shuttle in enumerate(shuttles):
    distance = first_shuttle.distance_from(shuttle)
    print(f"The first shuttle is {distance} units away from shuttle {i}.")

The first shuttle is 0.0 units away from shuttle 0.
The first shuttle is 57.94 units away from shuttle 1.
The first shuttle is 26.08 units away from shuttle 2.
The first shuttle is 38.9 units away from shuttle 3.
The first shuttle is 26.91 units away from shuttle 4.
The first shuttle is 35.9 units away from shuttle 5.
The first shuttle is 28.84 units away from shuttle 6.
The first shuttle is 38.21 units away from shuttle 7.
The first shuttle is 78.77 units away from shuttle 8.
The first shuttle is 67.88 units away from shuttle 9.
The first shuttle is 37.36 units away from shuttle 10.


In [21]:
for i, rocket in enumerate(rockets):
    distance = first_shuttle.distance_from(rocket)
    print(f"The first shuttle is {distance} units away from rocket {i}.")

The first shuttle is 15.62 units away from rocket 0.
The first shuttle is 58.6 units away from rocket 1.
The first shuttle is 29.83 units away from rocket 2.
The first shuttle is 42.06 units away from rocket 3.
The first shuttle is 98.41 units away from rocket 4.
The first shuttle is 38.08 units away from rocket 5.
The first shuttle is 66.1 units away from rocket 6.
The first shuttle is 2.24 units away from rocket 7.
The first shuttle is 76.24 units away from rocket 8.
The first shuttle is 61.2 units away from rocket 9.
The first shuttle is 22.67 units away from rocket 10.
